# Step 1 – Environment Setup & Basic RAG Pipeline

In this notebook, I implemented Step 1 of the RAG mastery task. My objective was to build a basic RAG pipeline that can load a real PDF document from Google Drive, split it into chunks, generate embeddings with Hugging Face `intfloat/multilingual-e5-large`, store the vectors in FAISS, retrieve relevant context, and generate a grounded answer using Vertex AI Gemini.

## Overall flow
`PDF from Drive → PDF loading → text validation → chunking → E5 embeddings → FAISS index → retrieval → prompt building → Vertex AI Gemini generation`

INPUT LIBS

In [1]:
!pip install langchain langchain-community langchain-huggingface langchain-google-vertexai faiss-cpu pypdf sentence-transformers

^C


  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached faiss_cpu-1.13.2-cp312-cp312-win_amd64.whl.metadata (7.6 kB)
  Using cached pypdf-6.11.0-py3-none-any.whl.metadata (7.2 kB)
  Using cached sentence_transformers-5.4.1-py3-none-any.whl.metadata (17 kB)
  Using cached langchain_core-1.4.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_protocol-0.0.15-py3-none-any.whl.metadata (2.4 kB)
  Using cached langsmith-0.8.3-py3-none-any.whl.metadata (15 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached uuid_utils-0.15.0-cp312-cp312-win_amd64.whl.metadata (6.7 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Us

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'D:\\INTERN\\venv\\Lib\\site-packages\\sklearn\\manifold\\__init__.py'
Check the permissions.



In [10]:
!pip install python-dotenv

LOAD SECRET KEY

In [2]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

if GOOGLE_API_KEY:
    print("Loaded GOOGLE_API_KEY successfully.")
else:
    print("GOOGLE_API_KEY not found.")

Loaded GOOGLE_API_KEY successfully.


In [1]:
import os
from dotenv import load_dotenv

# Tìm và nạp tự động các biến từ file .env vào hệ thống
load_dotenv()

# Truy xuất giá trị của khóa
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if GOOGLE_API_KEY:
    print("Đã nạp GOOGLE_API_KEY thành công bảo mật từ file .env!")
else:
    print("Không tìm thấy GOOGLE_API_KEY. Vui lòng kiểm tra lại file .env!")

Đã nạp GOOGLE_API_KEY thành công bảo mật từ file .env!


MOUTING TO DRIVE

In [2]:
# from google.colab import drive

# drive.mount('/content/drive')

GET THE TESTING DOC PATH

In [3]:
pdf_path = "../data/raws/RAG_(Retrieval-Augmented_Generation)_Mastery_RD_Guideline.pdf"

print("PDF path:", pdf_path)

PDF path: ../data/raws/RAG_(Retrieval-Augmented_Generation)_Mastery_RD_Guideline.pdf


## Cell 5 – Load the PDF Document

In this cell, I used `PyPDFLoader` to load the PDF file and extract its page-level text content. I also printed the number of loaded pages, the metadata of the first page, and a preview of the extracted text to confirm that the PDF was loaded correctly.

In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_path)
pages = loader.load()

print("Number of pages loaded:", len(pages))

print("\nFirst page metadata:")
print(pages[0].metadata)

print("\nFirst page preview:")
print(pages[0].page_content[:1000])

d:\Intern\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of pages loaded: 6

First page metadata:
{'producer': 'Skia/PDF m126', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/126.0.0.0 Safari/537.36', 'creationdate': '2026-05-05T08:36:21+00:00', 'title': 'RAG (Retrieval-Augmented Generation) Mastery R&D Guideline', 'moddate': '2026-05-05T08:36:21+00:00', 'source': '../data/raws/RAG_(Retrieval-Augmented_Generation)_Mastery_RD_Guideline.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}

First page preview:
RAG (Retrieval-Augmented
Generation) Mastery R&D
Guideline
Tags LLM RAG Vertex
RAG  Retrieval-Augmented Generation) Mastery R&D Guideline
Complete Step-by-Step Research, Actions & Milestones 
Career Impact Summary – Why This Challenge Matters for Your AI
Engineer Path
Main Goal Achievement in AI Engineer Career Path
Completing this guideline gives you end-to-end production mastery of Retrieval-
Augmented Generation  RAG) systems. You will learn data preprocessing,
embedding strate

## Cell 6 – Validate Extracted Text Content

In this cell, I checked whether the loaded PDF pages contain extractable text. This validation is important because a scanned PDF or image-based PDF may not provide usable text for RAG without OCR.

In [5]:
non_empty_pages = [page for page in pages if page.page_content.strip()]

print("Total pages:", len(pages))
print("Non-empty pages:", len(non_empty_pages))

if len(non_empty_pages) == 0:
    raise ValueError("PDF này có thể là PDF scan/ảnh. Cần OCR trước.")
else:
    print("PDF có text, có thể tiếp tục làm RAG.")

Total pages: 6
Non-empty pages: 6
PDF có text, có thể tiếp tục làm RAG.


## Cell 7 – Split the Document into Overlapping Text Chunks

In this cell, I split the PDF pages into smaller overlapping chunks using `RecursiveCharacterTextSplitter`. This step prepares the document for embedding and retrieval because the full PDF is too large to embed and retrieve as one single block.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(non_empty_pages)

print("Number of chunks:", len(chunks))

print("\nFirst chunk metadata:")
print(chunks[0].metadata)

print("\nFirst chunk content:")
print(chunks[0].page_content[:1000])

Number of chunks: 16

First chunk metadata:
{'producer': 'Skia/PDF m126', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/126.0.0.0 Safari/537.36', 'creationdate': '2026-05-05T08:36:21+00:00', 'title': 'RAG (Retrieval-Augmented Generation) Mastery R&D Guideline', 'moddate': '2026-05-05T08:36:21+00:00', 'source': '../data/raws/RAG_(Retrieval-Augmented_Generation)_Mastery_RD_Guideline.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}

First chunk content:
RAG (Retrieval-Augmented
Generation) Mastery R&D
Guideline
Tags LLM RAG Vertex
RAG  Retrieval-Augmented Generation) Mastery R&D Guideline
Complete Step-by-Step Research, Actions & Milestones 
Career Impact Summary – Why This Challenge Matters for Your AI
Engineer Path
Main Goal Achievement in AI Engineer Career Path
Completing this guideline gives you end-to-end production mastery of Retrieval-
Augmented Generation  RAG) systems. You will learn data preprocessing,
embedding strategie

## Cell 8 – Initialize Hugging Face E5-Large Embedding Model

In this cell, I created a custom LangChain embedding class for `intfloat/multilingual-e5-large`. I used the required E5 input prefixes: `passage:` for document chunks and `query:` for user questions. I also enabled normalized embeddings so that vector similarity search becomes more stable.

In [7]:
from langchain_core.embeddings import Embeddings
from sentence_transformers import SentenceTransformer
import torch

class E5LargeEmbeddings(Embeddings):
    def __init__(self, model_name="intfloat/multilingual-e5-large"):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Using device:", device)

        self.model = SentenceTransformer(
            model_name,
            device=device
        )

    def embed_documents(self, texts):
        texts = [f"passage: {text}" for text in texts]

        embeddings = self.model.encode(
            texts,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=True
        )

        return embeddings.tolist()

    def embed_query(self, text):
        text = f"query: {text}"

        embedding = self.model.encode(
            text,
            normalize_embeddings=True,
            convert_to_numpy=True
        )

        return embedding.tolist()


embeddings = E5LargeEmbeddings()

Using device: cpu


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6842.40it/s]


## Cell 9 – Build the FAISS Vector Index

In this cell, I embedded all document chunks and stored them in a FAISS vector store. FAISS allows the system to perform semantic similarity search and retrieve the most relevant chunks for a user query.

In [8]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("FAISS vector store created successfully.")

Batches: 100%|██████████| 1/1 [00:10<00:00, 10.37s/it]

FAISS vector store created successfully.


## Cell 10 – Test Top-K Semantic Retrieval

In this cell, I tested the retrieval stage by asking a sample question: `What is the milestone of Step 1?`. The system searches the FAISS index and returns the top relevant chunks from the PDF. This verifies that the embedding and retrieval components are working before connecting the generator model.

In [9]:
query = "What is the milestone of Step 1?"

retrieved_docs = vectorstore.similarity_search(
    query,
    k=4
)

print("Query:", query)

for i, doc in enumerate(retrieved_docs, start=1):
    print("=" * 80)
    print(f"Result {i}")
    print("Metadata:", doc.metadata)
    print(doc.page_content[:1000])

Query: What is the milestone of Step 1?
Result 1
Metadata: {'producer': 'Skia/PDF m126', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/126.0.0.0 Safari/537.36', 'creationdate': '2026-05-05T08:36:21+00:00', 'title': 'RAG (Retrieval-Augmented Generation) Mastery R&D Guideline', 'moddate': '2026-05-05T08:36:21+00:00', 'source': '../data/raws/RAG_(Retrieval-Augmented_Generation)_Mastery_RD_Guideline.pdf', 'total_pages': 6, 'page': 4, 'page_label': '5'}
Actions: Create complete report connecting all steps, including
database/embedding choices, evaluation results, and POC outcomes.
Milestone: Final portfolio  GitHub repo + HF Organization + live GCP POC + full
report).
RAG (Retrieval-Augmented Generation) Mastery R&D Guideline
5
Result 2
Metadata: {'producer': 'Skia/PDF m126', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/126.0.0.0 Safari/537.36', 'creationdate': '2026-05-05T08:36:21+00:00

## Cell 11 – Inspect Retrieval Scores

In this cell, I retrieved the top chunks together with their FAISS distance scores. This helps me inspect how close each retrieved chunk is to the query. In this FAISS setup, a lower distance score usually means the chunk is more similar to the query.

In [10]:
retrieved_docs_with_scores = vectorstore.similarity_search_with_score(
    query,
    k=4
)

for i, (doc, score) in enumerate(retrieved_docs_with_scores, start=1):
    print("=" * 80)
    print(f"Result {i}")
    print("FAISS distance score:", score)
    print("Page:", doc.metadata.get("page", "unknown"))
    print(doc.page_content[:800])

Result 1
FAISS distance score: 0.42181122
Page: 4
Actions: Create complete report connecting all steps, including
database/embedding choices, evaluation results, and POC outcomes.
Milestone: Final portfolio  GitHub repo + HF Organization + live GCP POC + full
report).
RAG (Retrieval-Augmented Generation) Mastery R&D Guideline
5
Result 2
FAISS distance score: 0.4229374
Page: 4
Step 11  Model & Data Management on GCP & HF
Research/Learn: Storing embeddings/datasets on Cloud Storage, versioning on
HF Hub, monitoring on Vertex AI.
Actions: Push processed datasets and models to HF Hub; set up GCP storage.
Milestone: Clean HF Organization + GCP bucket with all artifacts.
Step 12  Production Considerations on GCP
Research/Learn: Scaling, cost optimization, security, monitoring  Vertex AI RAG
Engine), logging, compliance.
Actions: Review deployment patterns and best practices.
Milestone: Production readiness checklist documented.
Step 13  POC Project – Build & Deploy on GCP
Research/Learn: End

## Cell 12 – Authenticate and Initialize Vertex AI Gemini

In this cell, I authenticated my Colab session with Google Cloud, selected the target GCP project, enabled the Vertex AI API, and initialized the Vertex AI Gemini client. This prepares the generator component of the RAG pipeline.

In [11]:
import os
from google import genai
from google.genai import types

# 1. Khai báo thông tin dự án
PROJECT_ID = "rag-project-495601"
LOCATION = "us-central1" # Khuyên dùng các region chuẩn hỗ trợ mạnh Vertex AI như us-central1 hoặc asia-southeast1

# 2. Thiết lập Client kết nối tường minh tới Vertex AI
try:
    vertex_client = genai.Client(
        vertexai=True,
        project=PROJECT_ID,
        location=LOCATION
    )
    print("🚀 Vertex AI Gemini client đã kết nối thành công trên Local!")
except Exception as e:
    print(f"❌ Lỗi kết nối. Vui lòng kiểm tra lại bước gcloud auth. Chi tiết lỗi: {e}")

🚀 Vertex AI Gemini client đã kết nối thành công trên Local!


## Cell 13 – Build the RAG Prompt from Retrieved Context

In this cell, I defined the prompt-building functions. The first function formats the retrieved PDF chunks with source and page information. The second function combines the retrieved context with the user question to create a grounded RAG prompt. This prompt instructs Gemini to answer only from the provided PDF context and avoid inventing unsupported facts.

In [12]:
def format_docs_for_prompt(docs):
    context_blocks = []

    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "PDF")
        page = doc.metadata.get("page", None)

        if isinstance(page, int):
            page_label = f"page {page + 1}"
        else:
            page_label = "unknown page"

        context_blocks.append(
            f"[Context {i} | Source: {source} | {page_label}]\n{doc.page_content}"
        )

    return "\n\n".join(context_blocks)


def build_rag_prompt(query, docs):
    context = format_docs_for_prompt(docs)

    prompt = f"""
You are a RAG assistant.

Answer the user's question using only the provided PDF context.
If the context does not contain enough information, say:
"The provided PDF does not contain enough information."

Do not invent facts.
Mention the relevant page when possible.

Context:
{context}

Question:
{query}

Answer:
"""

    return prompt

## Cell 14 – Generate an Answer with Vertex AI Gemini

In this cell, I defined a helper function that sends the completed RAG prompt to Vertex AI Gemini and returns the generated answer. At this stage, Vertex AI does not retrieve documents by itself; it only receives the prompt that I built from the retrieved context.

In [13]:
def generate_with_vertex_ai(prompt, model="gemini-2.5-flash"):
    response = vertex_client.models.generate_content(
        model=model,
        contents=prompt
    )

    return response.text

## Cell 15 – Create the Complete PDF RAG Function

In this cell, I combined retrieval, prompt construction, and Vertex AI generation into one complete function named `ask_pdf_rag`. The function first retrieves top-k relevant chunks from FAISS, then builds a context-grounded prompt, sends it to Vertex AI Gemini, and finally returns the query, retrieved documents, prompt, and generated answer.

In [14]:
def ask_pdf_rag(query, k=4, model="gemini-2.5-flash"):
    # 1. Retrieve top-k chunks từ FAISS
    retrieved_docs = vectorstore.similarity_search(
        query,
        k=k
    )

    # 2. Build prompt từ query + retrieved context
    prompt = build_rag_prompt(query, retrieved_docs)

    # 3. Gửi prompt sang Vertex AI Gemini
    answer = generate_with_vertex_ai(
        prompt=prompt,
        model=model
    )

    # 4. Trả về kết quả
    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "prompt": prompt,
        "answer": answer
    }

## Cell 16 – Run the End-to-End RAG Test

In this final test cell, I asked the system: `What is the milestone of Step 1?`. The notebook prints the original question, the full prompt sent to Vertex AI, and the final generated answer. This confirms that the full Step 1 pipeline works end to end: PDF loading, chunking, embedding, retrieval, prompt augmentation, and generation.

In [15]:
result = ask_pdf_rag(
    query="What is the milestone of Step 1?",
    k=4
)

print("QUESTION:")
print(result["query"])

print("\nPROMPT SENT TO VERTEX AI:")
print(result["prompt"])

print("\nANSWER:")
print(result["answer"])

QUESTION:
What is the milestone of Step 1?

PROMPT SENT TO VERTEX AI:

You are a RAG assistant.

Answer the user's question using only the provided PDF context.
If the context does not contain enough information, say:
"The provided PDF does not contain enough information."

Do not invent facts.
Mention the relevant page when possible.

Context:
[Context 1 | Source: ../data/raws/RAG_(Retrieval-Augmented_Generation)_Mastery_RD_Guideline.pdf | page 5]
Actions: Create complete report connecting all steps, including
database/embedding choices, evaluation results, and POC outcomes.
Milestone: Final portfolio  GitHub repo + HF Organization + live GCP POC + full
report).
RAG (Retrieval-Augmented Generation) Mastery R&D Guideline
5

[Context 2 | Source: ../data/raws/RAG_(Retrieval-Augmented_Generation)_Mastery_RD_Guideline.pdf | page 5]
Step 11  Model & Data Management on GCP & HF
Research/Learn: Storing embeddings/datasets on Cloud Storage, versioning on
HF Hub, monitoring on Vertex AI.
Action

## Step 1 Completion Summary

Based on the completed notebook, I successfully implemented the baseline RAG pipeline required for Step 1. The system can load a PDF from Google Drive, extract text, split it into chunks, generate multilingual E5-large embeddings, build a FAISS vector index, retrieve relevant chunks, construct a grounded RAG prompt, and generate an answer using Vertex AI Gemini. This satisfies the Step 1 milestone: testing a basic RAG pipeline.


In [16]:
folder_path = "../data/artifacts"
vectorstore.save_local(folder_path)
print(f"Đã lưu FAISS index tại: {folder_path}")

Đã lưu FAISS index tại: ../data/artifacts


# Step 2: Data Preprocessing Techniques for RAG

In this step, I focused on preprocessing techniques for RAG. The goal is to compare different chunking strategies and identify a suitable preprocessing pipeline for the next stages of the project.

The strategies tested in this step include:

1. Page-level chunking
2. Fixed-size character chunking
3. Recursive character chunking
4. Sentence-based chunking
5. Semantic chunking
6. Hierarchical chunking
7. LLM-based chunking

**Cell: Text Cleaning**

In this cell, I cleaned the extracted PDF text before applying chunking methods. The cleaning process removes unnecessary line breaks, normalizes spaces, and reduces formatting noise from the PDF extraction process.
This step is important because noisy text can negatively affect chunk quality, embedding quality, and retrieval accuracy.

In [17]:
import re
from langchain_core.documents import Document

def clean_text(text: str) -> str:
    """
    Clean extracted PDF text before chunking.
    This function removes formatting noise from PDF extraction.
    """
    if not text:
        return ""

    # Remove broken hyphen line breaks, e.g. "Retrieval-\nAugmented" -> "Retrieval-Augmented"
    text = re.sub(r"-\s*\n\s*", "-", text)

    # Replace multiple newlines with spaces
    text = re.sub(r"\n+", " ", text)

    # Remove control characters that may appear after PDF extraction
    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)

    # Normalize multiple spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()


cleaned_pages = []

for page in non_empty_pages:
    original_text = page.page_content
    cleaned_content = clean_text(original_text)

    if cleaned_content:
        cleaned_pages.append(
            Document(
                page_content=cleaned_content,
                metadata={
                    **page.metadata,
                    "preprocessing": "cleaned",
                    "cleaning_status": "completed",
                    "original_char_count": len(original_text),
                    "cleaned_char_count": len(cleaned_content),
                    "cleaned_word_count": len(cleaned_content.split())
                }
            )
        )

print("Original non-empty pages:", len(non_empty_pages))
print("Cleaned pages:", len(cleaned_pages))

print("\nPreview cleaned first page:")
print(cleaned_pages[0].page_content[:1000])

print("\nMetadata of cleaned first page:")
print(cleaned_pages[0].metadata)

Original non-empty pages: 6
Cleaned pages: 6

Preview cleaned first page:
RAG (Retrieval-Augmented Generation) Mastery R&D Guideline Tags LLM RAG Vertex RAG Retrieval-Augmented Generation) Mastery R&D Guideline Complete Step-by-Step Research, Actions & Milestones Career Impact Summary – Why This Challenge Matters for Your AI Engineer Path Main Goal Achievement in AI Engineer Career Path Completing this guideline gives you end-to-end production mastery of Retrieval-Augmented Generation RAG) systems. You will learn data preprocessing, embedding strategies, vector database selection (especially on GCP , advanced evaluation techniques (retrieval metrics + LLM-as-Judge), and how to build and deploy a full RAG pipeline. The result: you become a production-ready RAG Engineer who can solve real business problems with accurate, grounded, hallucination-resistant AI using open and cloud-native tools. Why This Is Important High Industry Demand: In 2026, almost every company building LLM applicatio

## Cell: Metadata Extraction

In this cell, I enriched each cleaned PDF page with structured metadata before applying chunking strategies. The added metadata includes the source file name, file path, page index, page number, document type, character count, word count, and preprocessing stage. This helps support source tracking, citation, filtering, and later evaluation in the RAG pipeline.

In [18]:
from pathlib import Path
from langchain_core.documents import Document

pdf_name = Path(pdf_path).name

def enrich_page_metadata(doc, page_index):
    page_number = doc.metadata.get("page", page_index)

    return Document(
        page_content=doc.page_content,
        metadata={
            **doc.metadata,
            "source_file": pdf_name,
            "source_path": pdf_path,
            "page_index": page_index,
            "page_number": page_number + 1 if isinstance(page_number, int) else page_index + 1,
            "document_type": "pdf",
            "pipeline_stage": "step2_preprocessing",
            "char_count": len(doc.page_content),
            "word_count": len(doc.page_content.split())
        }
    )

metadata_pages = [
    enrich_page_metadata(doc, i)
    for i, doc in enumerate(cleaned_pages)
]

documents_for_chunking = metadata_pages

print("Metadata-enriched pages:", len(metadata_pages))
print("\nSample metadata:")
print(metadata_pages[0].metadata)

Metadata-enriched pages: 6

Sample metadata:
{'producer': 'Skia/PDF m126', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/126.0.0.0 Safari/537.36', 'creationdate': '2026-05-05T08:36:21+00:00', 'title': 'RAG (Retrieval-Augmented Generation) Mastery R&D Guideline', 'moddate': '2026-05-05T08:36:21+00:00', 'source': '../data/raws/RAG_(Retrieval-Augmented_Generation)_Mastery_RD_Guideline.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'preprocessing': 'cleaned', 'cleaning_status': 'completed', 'original_char_count': 1377, 'cleaned_char_count': 1370, 'cleaned_word_count': 188, 'source_file': 'RAG_(Retrieval-Augmented_Generation)_Mastery_RD_Guideline.pdf', 'source_path': '../data/raws/RAG_(Retrieval-Augmented_Generation)_Mastery_RD_Guideline.pdf', 'page_index': 0, 'page_number': 1, 'document_type': 'pdf', 'pipeline_stage': 'step2_preprocessing', 'char_count': 1370, 'word_count': 188}


**Cell: Page-Level Chunking**

In this cell, I tested page-level chunking. Each PDF page is treated as one chunk. This method preserves page-level context and makes it easy to cite the original source page.

However, page-level chunks may become too long if a page contains a lot of text, which can reduce retrieval precision.

In [19]:
page_level_chunks = []

for i, page in enumerate(documents_for_chunking):
    page_level_chunks.append(
        Document(
            page_content=page.page_content,
            metadata={
                **page.metadata,
                "chunk_strategy": "page_level",
                "chunk_id": f"page_level_{i}"
            }
        )
    )

print("Number of page-level chunks:", len(page_level_chunks))
print("\nFirst page-level chunk:")
print(page_level_chunks[0].page_content[:1000])

Number of page-level chunks: 6

First page-level chunk:
RAG (Retrieval-Augmented Generation) Mastery R&D Guideline Tags LLM RAG Vertex RAG Retrieval-Augmented Generation) Mastery R&D Guideline Complete Step-by-Step Research, Actions & Milestones Career Impact Summary – Why This Challenge Matters for Your AI Engineer Path Main Goal Achievement in AI Engineer Career Path Completing this guideline gives you end-to-end production mastery of Retrieval-Augmented Generation RAG) systems. You will learn data preprocessing, embedding strategies, vector database selection (especially on GCP , advanced evaluation techniques (retrieval metrics + LLM-as-Judge), and how to build and deploy a full RAG pipeline. The result: you become a production-ready RAG Engineer who can solve real business problems with accurate, grounded, hallucination-resistant AI using open and cloud-native tools. Why This Is Important High Industry Demand: In 2026, almost every company building LLM applications uses RAG for co

## Cell: Fixed-Size Character Chunking

In this cell, I tested fixed-size character chunking. The text is divided into chunks with a fixed character length and a defined overlap between consecutive chunks.

This method is simple and easy to control, but it may cut sentences or ideas in the middle because it does not understand document structure.

In [20]:
def fixed_size_chunk_documents(documents, chunk_size=800, chunk_overlap=120):
    fixed_chunks = []
    step = chunk_size - chunk_overlap

    chunk_counter = 0

    for doc in documents:
        text = doc.page_content

        for start in range(0, len(text), step):
            end = start + chunk_size
            chunk_text = text[start:end].strip()

            if chunk_text:
                fixed_chunks.append(
                    Document(
                        page_content=chunk_text,
                        metadata={
                            **doc.metadata,
                            "chunk_strategy": "fixed_size",
                            "chunk_id": f"fixed_size_{chunk_counter}",
                            "start_char": start,
                            "end_char": end
                        }
                    )
                )

                chunk_counter += 1

    return fixed_chunks


fixed_size_chunks = fixed_size_chunk_documents(
    documents_for_chunking,
    chunk_size=800,
    chunk_overlap=120
)

print("Number of fixed-size chunks:", len(fixed_size_chunks))
print("\nFirst fixed-size chunk:")
print(fixed_size_chunks[0].page_content[:1000])

Number of fixed-size chunks: 17

First fixed-size chunk:
RAG (Retrieval-Augmented Generation) Mastery R&D Guideline Tags LLM RAG Vertex RAG Retrieval-Augmented Generation) Mastery R&D Guideline Complete Step-by-Step Research, Actions & Milestones Career Impact Summary – Why This Challenge Matters for Your AI Engineer Path Main Goal Achievement in AI Engineer Career Path Completing this guideline gives you end-to-end production mastery of Retrieval-Augmented Generation RAG) systems. You will learn data preprocessing, embedding strategies, vector database selection (especially on GCP , advanced evaluation techniques (retrieval metrics + LLM-as-Judge), and how to build and deploy a full RAG pipeline. The result: you become a production-ready RAG Engineer who can solve real business problems with accurate, grounded, hallucination-resistant AI using


## Cell: Recursive Character Chunking

In this cell, I tested recursive character chunking. This method attempts to split text using larger separators first, such as paragraphs and lines, before falling back to smaller units like sentences, words, or characters.

Compared with fixed-size chunking, recursive chunking usually preserves the structure and meaning of the document better. This is a strong candidate for the final preprocessing pipeline.

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""],
    add_start_index=True
)

recursive_chunks = recursive_splitter.split_documents(documents_for_chunking)

for i, chunk in enumerate(recursive_chunks):
    chunk.metadata["chunk_strategy"] = "recursive_character"
    chunk.metadata["chunk_id"] = f"recursive_character_{i}"

print("Number of recursive character chunks:", len(recursive_chunks))
print("\nFirst recursive character chunk:")
print(recursive_chunks[0].page_content[:1000])

Number of recursive character chunks: 16

First recursive character chunk:
RAG (Retrieval-Augmented Generation) Mastery R&D Guideline Tags LLM RAG Vertex RAG Retrieval-Augmented Generation) Mastery R&D Guideline Complete Step-by-Step Research, Actions & Milestones Career Impact Summary – Why This Challenge Matters for Your AI Engineer Path Main Goal Achievement in AI Engineer Career Path Completing this guideline gives you end-to-end production mastery of Retrieval-Augmented Generation RAG) systems. You will learn data preprocessing, embedding strategies, vector database selection (especially on GCP , advanced evaluation techniques (retrieval metrics + LLM-as-Judge), and how to build and deploy a full RAG pipeline


## Cell: Sentence-Based Chunking

In this cell, I tested sentence-based chunking. The text is first split into sentences, then grouped into chunks with a maximum character length.

This method helps avoid cutting sentences in the middle, so each chunk is often more readable. However, it depends on the quality of sentence boundary detection and may not work perfectly for complex PDF formatting.

In [22]:
def sentence_based_chunk_documents(documents, max_chars=800, overlap_sentences=1):
    sentence_chunks = []
    chunk_counter = 0

    for doc in documents:
        text = doc.page_content

        # Simple sentence splitting
        sentences = re.split(r"(?<=[.!?])\s+", text)

        current_chunk_sentences = []

        for sentence in sentences:
            sentence = sentence.strip()

            if not sentence:
                continue

            candidate_chunk = " ".join(current_chunk_sentences + [sentence])

            if len(candidate_chunk) <= max_chars:
                current_chunk_sentences.append(sentence)
            else:
                if current_chunk_sentences:
                    chunk_text = " ".join(current_chunk_sentences)

                    sentence_chunks.append(
                        Document(
                            page_content=chunk_text,
                            metadata={
                                **doc.metadata,
                                "chunk_strategy": "sentence_based",
                                "chunk_id": f"sentence_based_{chunk_counter}"
                            }
                        )
                    )

                    chunk_counter += 1

                    # Keep sentence overlap
                    current_chunk_sentences = current_chunk_sentences[-overlap_sentences:] if overlap_sentences > 0 else []

                current_chunk_sentences.append(sentence)

        if current_chunk_sentences:
            chunk_text = " ".join(current_chunk_sentences)

            sentence_chunks.append(
                Document(
                    page_content=chunk_text,
                    metadata={
                        **doc.metadata,
                        "chunk_strategy": "sentence_based",
                        "chunk_id": f"sentence_based_{chunk_counter}"
                    }
                )
            )

            chunk_counter += 1

    return sentence_chunks


sentence_based_chunks = sentence_based_chunk_documents(
    documents_for_chunking,
    max_chars=800,
    overlap_sentences=1
)

print("Number of sentence-based chunks:", len(sentence_based_chunks))
print("\nFirst sentence-based chunk:")
print(sentence_based_chunks[0].page_content[:1000])

Number of sentence-based chunks: 17

First sentence-based chunk:
RAG (Retrieval-Augmented Generation) Mastery R&D Guideline Tags LLM RAG Vertex RAG Retrieval-Augmented Generation) Mastery R&D Guideline Complete Step-by-Step Research, Actions & Milestones Career Impact Summary – Why This Challenge Matters for Your AI Engineer Path Main Goal Achievement in AI Engineer Career Path Completing this guideline gives you end-to-end production mastery of Retrieval-Augmented Generation RAG) systems. You will learn data preprocessing, embedding strategies, vector database selection (especially on GCP , advanced evaluation techniques (retrieval metrics + LLM-as-Judge), and how to build and deploy a full RAG pipeline.


**Cell: Semantic Chunking**

In this cell, I implemented semantic chunking. Instead of splitting the document only by character length, I used sentence embeddings to detect semantic shifts between consecutive sentences.

When the semantic similarity between two adjacent sentences becomes low, the text is split into a new chunk.

In [23]:
import re
import numpy as np
from langchain_core.documents import Document

def split_into_sentences(text):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if s.strip()]


def semantic_chunk_documents(
    documents,
    embedder,
    max_chars=1000,
    similarity_percentile=25
):
    semantic_chunks = []
    chunk_counter = 0

    for doc in documents:
        sentences = split_into_sentences(doc.page_content)

        if len(sentences) <= 1:
            semantic_chunks.append(
                Document(
                    page_content=doc.page_content,
                    metadata={
                        **doc.metadata,
                        "chunk_strategy": "semantic",
                        "chunk_id": f"semantic_{chunk_counter}"
                    }
                )
            )
            chunk_counter += 1
            continue

        sentence_inputs = [f"passage: {sentence}" for sentence in sentences]

        sentence_embeddings = embedder.model.encode(
            sentence_inputs,
            normalize_embeddings=True,
            convert_to_numpy=True
        )

        similarities = []

        for i in range(len(sentence_embeddings) - 1):
            sim = np.dot(sentence_embeddings[i], sentence_embeddings[i + 1])
            similarities.append(sim)

        threshold = np.percentile(similarities, similarity_percentile)

        current_sentences = [sentences[0]]

        for i, sim in enumerate(similarities):
            next_sentence = sentences[i + 1]
            candidate_text = " ".join(current_sentences + [next_sentence])

            if sim < threshold or len(candidate_text) > max_chars:
                chunk_text = " ".join(current_sentences).strip()

                if chunk_text:
                    semantic_chunks.append(
                        Document(
                            page_content=chunk_text,
                            metadata={
                                **doc.metadata,
                                "chunk_strategy": "semantic",
                                "chunk_id": f"semantic_{chunk_counter}"
                            }
                        )
                    )
                    chunk_counter += 1

                current_sentences = [next_sentence]
            else:
                current_sentences.append(next_sentence)

        if current_sentences:
            chunk_text = " ".join(current_sentences).strip()

            semantic_chunks.append(
                Document(
                    page_content=chunk_text,
                    metadata={
                        **doc.metadata,
                        "chunk_strategy": "semantic",
                        "chunk_id": f"semantic_{chunk_counter}"
                    }
                )
            )
            chunk_counter += 1

    return semantic_chunks


semantic_chunks = semantic_chunk_documents(
    documents=documents_for_chunking,
    embedder=embeddings,
    max_chars=1000,
    similarity_percentile=25
)

print("Number of semantic chunks:", len(semantic_chunks))
print("\nFirst semantic chunk:")
print(semantic_chunks[0].page_content[:1000])

Number of semantic chunks: 24

First semantic chunk:
RAG (Retrieval-Augmented Generation) Mastery R&D Guideline Tags LLM RAG Vertex RAG Retrieval-Augmented Generation) Mastery R&D Guideline Complete Step-by-Step Research, Actions & Milestones Career Impact Summary – Why This Challenge Matters for Your AI Engineer Path Main Goal Achievement in AI Engineer Career Path Completing this guideline gives you end-to-end production mastery of Retrieval-Augmented Generation RAG) systems. You will learn data preprocessing, embedding strategies, vector database selection (especially on GCP , advanced evaluation techniques (retrieval metrics + LLM-as-Judge), and how to build and deploy a full RAG pipeline. The result: you become a production-ready RAG Engineer who can solve real business problems with accurate, grounded, hallucination-resistant AI using open and cloud-native tools. Why This Is Important High Industry Demand: In 2026, almost every company building LLM applications uses RAG for cost,

**Cell: Hierarchical Chunking**

English:
In this cell, I implemented hierarchical chunking. The document was first divided into larger parent chunks to preserve broader context.

Each parent chunk was then divided into smaller child chunks for more precise retrieval. Each child chunk keeps a parent_id so that the system can trace it back to its broader context.

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
    add_start_index=True
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""],
    add_start_index=True
)

parent_chunks = parent_splitter.split_documents(documents_for_chunking)

hierarchical_child_chunks = []

for parent_idx, parent_doc in enumerate(parent_chunks):
    parent_id = f"parent_{parent_idx}"

    parent_doc.metadata["chunk_strategy"] = "hierarchical_parent"
    parent_doc.metadata["parent_id"] = parent_id

    child_docs = child_splitter.split_documents([parent_doc])

    for child_idx, child_doc in enumerate(child_docs):
        child_doc.metadata["chunk_strategy"] = "hierarchical_child"
        child_doc.metadata["parent_id"] = parent_id
        child_doc.metadata["child_id"] = f"{parent_id}_child_{child_idx}"

        hierarchical_child_chunks.append(child_doc)

print("Number of parent chunks:", len(parent_chunks))
print("Number of hierarchical child chunks:", len(hierarchical_child_chunks))

print("\nFirst parent chunk:")
print(parent_chunks[0].page_content[:700])

print("\nFirst child chunk:")
print(hierarchical_child_chunks[0].page_content[:700])
print("\nChild metadata:")
print(hierarchical_child_chunks[0].metadata)

Number of parent chunks: 6
Number of hierarchical child chunks: 26

First parent chunk:
RAG (Retrieval-Augmented Generation) Mastery R&D Guideline Tags LLM RAG Vertex RAG Retrieval-Augmented Generation) Mastery R&D Guideline Complete Step-by-Step Research, Actions & Milestones Career Impact Summary – Why This Challenge Matters for Your AI Engineer Path Main Goal Achievement in AI Engineer Career Path Completing this guideline gives you end-to-end production mastery of Retrieval-Augmented Generation RAG) systems. You will learn data preprocessing, embedding strategies, vector database selection (especially on GCP , advanced evaluation techniques (retrieval metrics + LLM-as-Judge), and how to build and deploy a full RAG pipeline. The result: you become a production-ready RAG Engi

First child chunk:
RAG (Retrieval-Augmented Generation) Mastery R&D Guideline Tags LLM RAG Vertex RAG Retrieval-Augmented Generation) Mastery R&D Guideline Complete Step-by-Step Research, Actions & Milestones C

**Cell: LLM-Based Chunking with Vertex AI**

In this cell, I implemented LLM-based chunking using Vertex AI Gemini. The model was prompted to split the document text into coherent semantic chunks while preserving the original meaning and wording. I limited the experiment to a small number of pages to control API cost and latency.

In [25]:
import json
import re
from langchain_core.documents import Document

def parse_json_array_from_llm(response_text):
    """
    Parse a JSON array returned by the LLM.
    This function handles both raw JSON arrays and markdown-wrapped JSON blocks.
    """
    if not response_text:
        return None

    cleaned = response_text.strip()

    # Remove markdown code fences if the model returns ```json ... ```
    cleaned = re.sub(r"^```json\s*", "", cleaned)
    cleaned = re.sub(r"^```\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    # Try direct JSON parsing
    try:
        result = json.loads(cleaned)
        if isinstance(result, list):
            return result
    except Exception:
        pass

    # Try extracting the first JSON array from the response
    match = re.search(r"\[.*\]", cleaned, re.DOTALL)

    if match:
        try:
            result = json.loads(match.group(0))
            if isinstance(result, list):
                return result
        except Exception:
            pass

    return None


def llm_chunk_single_text(text, model="gemini-2.5-flash"):
    prompt = f"""
You are a document preprocessing assistant for a RAG system.

Your task is to split the following text into coherent semantic chunks.

Rules:
- Each chunk should contain one main idea.
- Do not summarize the text.
- Do not add new information.
- Preserve the original wording as much as possible.
- Return only a valid JSON array of strings.
- Each string should be one chunk.
- Do not wrap the JSON array in markdown code fences.

Text:
{text}

JSON array:
"""

    response_text = generate_with_vertex_ai(
        prompt=prompt,
        model=model
    )

    parsed_chunks = parse_json_array_from_llm(response_text)

    if parsed_chunks:
        return [
            chunk.strip()
            for chunk in parsed_chunks
            if isinstance(chunk, str) and chunk.strip()
        ]

    # Fallback if the LLM does not return valid JSON
    return [text]


def llm_based_chunk_documents(documents, max_pages=3):
    llm_chunks = []
    chunk_counter = 0

    # Limit the number of pages to control API cost and latency
    for doc in documents[:max_pages]:
        generated_chunks = llm_chunk_single_text(doc.page_content)

        for chunk_text in generated_chunks:
            llm_chunks.append(
                Document(
                    page_content=chunk_text,
                    metadata={
                        **doc.metadata,
                        "chunk_strategy": "llm_based",
                        "chunk_id": f"llm_based_{chunk_counter}"
                    }
                )
            )
            chunk_counter += 1

    return llm_chunks


llm_based_chunks = llm_based_chunk_documents(
    documents=documents_for_chunking,
    max_pages=3
)

print("Number of LLM-based chunks:", len(llm_based_chunks))

if len(llm_based_chunks) > 0:
    print("\nFirst LLM-based chunk:")
    print(llm_based_chunks[0].page_content[:1000])
else:
    print("No LLM-based chunks were generated.")

Number of LLM-based chunks: 30

First LLM-based chunk:
RAG (Retrieval-Augmented Generation) Mastery R&D Guideline Tags LLM RAG Vertex RAG Retrieval-Augmented Generation) Mastery R&D Guideline Complete Step-by-Step Research, Actions & Milestones


## Cell: Chunking Statistics Summary

In this cell, I summarized the basic statistics of all chunking strategies, including the number of chunks, average character length, minimum and maximum length, and word count.

This helps compare how each chunking method affects the structure and size of the generated chunks.

In [26]:
import pandas as pd
import numpy as np

def summarize_chunks(strategy_name, chunks):
    if chunks is None or len(chunks) == 0:
        return {
            "strategy": strategy_name,
            "num_chunks": 0,
            "avg_chars": 0,
            "min_chars": 0,
            "max_chars": 0,
            "avg_words": 0,
            "min_words": 0,
            "max_words": 0,
        }

    lengths = [len(chunk.page_content) for chunk in chunks]
    word_counts = [len(chunk.page_content.split()) for chunk in chunks]

    return {
        "strategy": strategy_name,
        "num_chunks": len(chunks),
        "avg_chars": round(np.mean(lengths), 2),
        "min_chars": int(np.min(lengths)),
        "max_chars": int(np.max(lengths)),
        "avg_words": round(np.mean(word_counts), 2),
        "min_words": int(np.min(word_counts)),
        "max_words": int(np.max(word_counts)),
    }
chunking_strategies = {
    "page_level": page_level_chunks,
    "fixed_size": fixed_size_chunks,
    "recursive_character": recursive_chunks,
    "sentence_based": sentence_based_chunks,
    "semantic": semantic_chunks,
    "hierarchical_child": hierarchical_child_chunks,
    "llm_based": llm_based_chunks,
}
chunking_summary = pd.DataFrame([
    summarize_chunks(strategy_name, chunks)
    for strategy_name, chunks in chunking_strategies.items()
])

chunking_summary

chunking_summary

,strategy,num_chunks,avg_chars,min_chars,max_chars,avg_words,min_words,max_words
0,page_level,6,1614.67,1129,1876,216.50,148,255
1,fixed_size,17,640.76,10,800,85.65,2,114
2,recursive_character,16,623.25,280,778,84.19,38,111
3,sentence_based,17,657.35,394,800,88.65,52,111
4,semantic,24,402.92,7,975,54.12,1,134
5,hierarchical_child,26,386.12,62,498,52.46,8,72
6,llm_based,30,167.70,21,488,22.97,4,70


## Cell: Pros and Cons Comparison

In this cell, I documented the strengths, limitations, and suitable use cases of each chunking strategy.

This comparison supports the decision-making process for selecting the final preprocessing pipeline.

In [27]:
pros_cons_table = pd.DataFrame([
    {
        "strategy": "page_level",
        "pros": "Preserves full page context and makes page citation easy.",
        "cons": "Chunks may be too long and less precise for retrieval.",
        "best_use_case": "Short PDF pages or documents where page-level context is important."
    },
    {
        "strategy": "fixed_size",
        "pros": "Simple, consistent, and easy to control with chunk size and overlap.",
        "cons": "May cut sentences or ideas in the middle.",
        "best_use_case": "Baseline experiments and simple text documents."
    },
    {
        "strategy": "recursive_character",
        "pros": "Balances chunk size while preserving paragraph and sentence structure better.",
        "cons": "Still depends on separator choices and chunk size configuration.",
        "best_use_case": "General-purpose RAG preprocessing for PDFs and text documents."
    },
    {
        "strategy": "sentence_based",
        "pros": "Keeps sentences intact and improves readability of chunks.",
        "cons": "Sentence boundary detection may be imperfect for noisy PDF text.",
        "best_use_case": "Documents with well-formed sentences and clear punctuation."
    },
    {
        "strategy": "semantic",
        "pros": "Splits text based on semantic shifts, producing more meaning-aware chunks.",
        "cons": "Requires embedding computation and threshold tuning.",
        "best_use_case": "Documents with topic changes where semantic coherence is important."
    },
    {
        "strategy": "hierarchical_child",
        "pros": "Combines precise child-level retrieval with broader parent-level context.",
        "cons": "More complex metadata management and retrieval logic.",
        "best_use_case": "Long documents where both precision and broader context are needed."
    },
    {
        "strategy": "llm_based",
        "pros": "Can create highly coherent chunks based on meaning and document structure.",
        "cons": "Higher cost, higher latency, and less deterministic output.",
        "best_use_case": "High-value documents where chunk quality is more important than speed or cost."
    },
])

pros_cons_table

,strategy,pros,cons,best_use_case
0,page_level,Preserves full page context and makes page cit...,Chunks may be too long and less precise for re...,Short PDF pages or documents where page-level ...
1,fixed_size,"Simple, consistent, and easy to control with c...",May cut sentences or ideas in the middle.,Baseline experiments and simple text documents.
2,recursive_character,Balances chunk size while preserving paragraph...,Still depends on separator choices and chunk s...,General-purpose RAG preprocessing for PDFs and...
3,sentence_based,Keeps sentences intact and improves readabilit...,Sentence boundary detection may be imperfect f...,Documents with well-formed sentences and clear...
4,semantic,"Splits text based on semantic shifts, producin...",Requires embedding computation and threshold t...,Documents with topic changes where semantic co...
5,hierarchical_child,Combines precise child-level retrieval with br...,More complex metadata management and retrieval...,Long documents where both precision and broade...
6,llm_based,Can create highly coherent chunks based on mea...,"Higher cost, higher latency, and less determin...",High-value documents where chunk quality is mo...


## Cell: Qualitative Retrieval Test

In this cell, I tested each chunking strategy using the same query and FAISS retrieval.

This provides an initial qualitative check of whether each strategy can retrieve relevant context before formal retrieval metrics are implemented in later steps.

In [28]:
from langchain_community.vectorstores import FAISS

test_query = "What is the milestone of Step 1?"

retrieval_results = {}

for strategy_name, strategy_chunks in chunking_strategies.items():
    print("=" * 100)
    print("Strategy:", strategy_name)

    if len(strategy_chunks) == 0:
        print("No chunks available for this strategy.")
        continue

    temp_vectorstore = FAISS.from_documents(
        documents=strategy_chunks,
        embedding=embeddings
    )

    docs_with_scores = temp_vectorstore.similarity_search_with_score(
        test_query,
        k=3
    )

    retrieval_results[strategy_name] = docs_with_scores

    for i, (doc, score) in enumerate(docs_with_scores, start=1):
        page = doc.metadata.get("page", None)
        page_label = page + 1 if isinstance(page, int) else "unknown"

        print("-" * 80)
        print(f"Top {i} | FAISS distance score: {score:.4f} | Page: {page_label}")
        print(doc.page_content[:500])

Strategy: page_level


Batches: 100%|██████████| 1/1 [00:08<00:00,  8.05s/it]


--------------------------------------------------------------------------------
Top 1 | FAISS distance score: 0.4263 | Page: 5
Step 11 Model & Data Management on GCP & HF Research/Learn: Storing embeddings/datasets on Cloud Storage, versioning on HF Hub, monitoring on Vertex AI. Actions: Push processed datasets and models to HF Hub; set up GCP storage. Milestone: Clean HF Organization + GCP bucket with all artifacts. Step 12 Production Considerations on GCP Research/Learn: Scaling, cost optimization, security, monitoring Vertex AI RAG Engine), logging, compliance. Actions: Review deployment patterns and best practices. 
--------------------------------------------------------------------------------
Top 2 | FAISS distance score: 0.4439 | Page: 4
limits. Actions: Build end-to-end RAG (query → retrieve → augment → generate). Milestone: Functional RAG system answering questions on your test dataset. Step 7 RAG Evaluation – Retrieval Metrics Research/Learn: Curating test cases (ground-tru

Batches: 100%|██████████| 1/1 [00:11<00:00, 11.41s/it]


--------------------------------------------------------------------------------
Top 1 | FAISS distance score: 0.4147 | Page: 5
actices, cost analysis, versioning strategy. Actions: Create complete report connecting all steps, including database/embedding choices, evaluation results, and POC outcomes. Milestone: Final portfolio GitHub repo + HF Organization + live GCP POC + full report). RAG (Retrieval-Augmented Generation) Mastery R&D Guideline 5
--------------------------------------------------------------------------------
Top 2 | FAISS distance score: 0.4262 | Page: 5
Step 11 Model & Data Management on GCP & HF Research/Learn: Storing embeddings/datasets on Cloud Storage, versioning on HF Hub, monitoring on Vertex AI. Actions: Push processed datasets and models to HF Hub; set up GCP storage. Milestone: Clean HF Organization + GCP bucket with all artifacts. Step 12 Production Considerations on GCP Research/Learn: Scaling, cost optimization, security, monitoring Vertex AI RAG Engine

...

--------------------------------------------------------------------------------
Top 1 | FAISS distance score: 0.3878 | Page: 2
Step-by-Step Research, Actions & Milestones Follow these steps in sequence. Each step includes What to Research/Learn, Actions to Take, and Milestone/Deliverable. Step 1 Environment Setup & RAG Fundamentals Research/Learn: RAG architecture (retriever + generator), LangChain/LlamaIndex/Haystack basics, GCP Vertex AI Generative AI and RAG Engine, HF datasets. Actions: Set up GCP project Vertex AI, Cloud Storage, Colab), create HF account, install core libraries. Milestone: Successful test of a bas
--------------------------------------------------------------------------------
Top 2 | FAISS distance score: 0.4222 | Page: 5
Deploy as a Vertex AI endpoint or interactive Gradio/Streamlit app on Cloud Run for household users.Milestone: Live POC RAG application on GCP with evaluation report and demo link. Step 14 Final Integration & Documentation Research/Learn: End-

Batches: 100%|██████████| 1/1 [00:10<00:00, 10.91s/it]


--------------------------------------------------------------------------------
Top 1 | FAISS distance score: 0.3741 | Page: 2
. Milestone: Successful test of a basic naive RAG pipeline (load document → embed → retrieve → generate). Step 2 Data Preprocessing Techniques for RAG RAG (Retrieval-Augmented Generation) Mastery R&D Guideline 2
--------------------------------------------------------------------------------
Top 2 | FAISS distance score: 0.4057 | Page: 3
. Milestone: Working retriever that returns top-K chunks for sample queries. Step 6 Full RAG Pipeline Implementation Research/Learn: Integration of retriever + LLM Gemini on Vertex AI or open models), prompt engineering for grounded generation, handling context window RAG (Retrieval-Augmented Generation) Mastery R&D Guideline 3
--------------------------------------------------------------------------------
Top 3 | FAISS distance score: 0.4057 | Page: 2
. Step-by-Step Research, Actions & Milestones Follow these steps in sequen

Batches: 100%|██████████| 1/1 [00:12<00:00, 12.04s/it]


--------------------------------------------------------------------------------
Top 1 | FAISS distance score: 0.3495 | Page: 2
Milestone: Successful test of a basic naive RAG pipeline (load document → embed → retrieve → generate).
--------------------------------------------------------------------------------
Top 2 | FAISS distance score: 0.3652 | Page: 2
Each step includes What to Research/Learn, Actions to Take, and Milestone/Deliverable.
--------------------------------------------------------------------------------
Top 3 | FAISS distance score: 0.3958 | Page: 2
Step-by-Step Research, Actions & Milestones Follow these steps in sequence.


## Cell: Overlap Strategy Experiment

In this cell, I tested multiple chunk overlap values for recursive character chunking.
The purpose is to observe how overlap affects the number of chunks, context preservation, and possible redundancy.

In [29]:
overlap_values = [0, 80, 120, 200]

overlap_experiment_results = {}
overlap_summary_rows = []

for overlap in overlap_values:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
        add_start_index=True
    )

    chunks_for_overlap = splitter.split_documents(documents_for_chunking)

    for i, chunk in enumerate(chunks_for_overlap):
        chunk.metadata["chunk_strategy"] = "recursive_character"
        chunk.metadata["chunk_size"] = 800
        chunk.metadata["chunk_overlap"] = overlap
        chunk.metadata["chunk_id"] = f"recursive_overlap_{overlap}_{i}"

    overlap_experiment_results[overlap] = chunks_for_overlap

    lengths = [len(chunk.page_content) for chunk in chunks_for_overlap]
    word_counts = [len(chunk.page_content.split()) for chunk in chunks_for_overlap]

    overlap_summary_rows.append({
        "chunk_size": 800,
        "chunk_overlap": overlap,
        "num_chunks": len(chunks_for_overlap),
        "avg_chars": round(np.mean(lengths), 2),
        "min_chars": int(np.min(lengths)),
        "max_chars": int(np.max(lengths)),
        "avg_words": round(np.mean(word_counts), 2),
        "min_words": int(np.min(word_counts)),
        "max_words": int(np.max(word_counts)),
    })

overlap_summary = pd.DataFrame(overlap_summary_rows)

overlap_summary

,chunk_size,chunk_overlap,num_chunks,avg_chars,min_chars,max_chars,avg_words,min_words,max_words
0,800,0,16,605.50,280,791,81.81,38,111
1,800,80,16,617.81,280,778,83.50,38,111
2,800,120,16,623.25,280,778,84.19,38,111
3,800,200,16,665.56,304,784,90.12,38,111


## Cell: Final Preprocessing Pipeline Selection

In this cell, I selected recursive character chunking with chunk_size=800 and chunk_overlap=120 as the final preprocessing pipeline. I selected this strategy because it provides a good balance between chunk size control, context preservation, stability, and reproducibility.

In [30]:
selected_strategy_name = "recursive_character"
selected_chunk_size = 800
selected_chunk_overlap = 120

final_splitter = RecursiveCharacterTextSplitter(
    chunk_size=selected_chunk_size,
    chunk_overlap=selected_chunk_overlap,
    separators=["\n\n", "\n", ". ", " ", ""],
    add_start_index=True
)

final_processed_chunks = final_splitter.split_documents(documents_for_chunking)

for i, chunk in enumerate(final_processed_chunks):
    chunk.metadata["chunk_strategy"] = selected_strategy_name
    chunk.metadata["chunk_size"] = selected_chunk_size
    chunk.metadata["chunk_overlap"] = selected_chunk_overlap
    chunk.metadata["chunk_id"] = f"final_recursive_{i}"
    chunk.metadata["pipeline_status"] = "final_selected"

print("Selected preprocessing pipeline:")
print("Strategy:", selected_strategy_name)
print("Chunk size:", selected_chunk_size)
print("Chunk overlap:", selected_chunk_overlap)
print("Number of final chunks:", len(final_processed_chunks))

print("\nSample final chunk metadata:")
print(final_processed_chunks[0].metadata)

print("\nSample final chunk content:")
print(final_processed_chunks[0].page_content[:1000])

Selected preprocessing pipeline:
Strategy: recursive_character
Chunk size: 800
Chunk overlap: 120
Number of final chunks: 16

Sample final chunk metadata:
{'producer': 'Skia/PDF m126', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/126.0.0.0 Safari/537.36', 'creationdate': '2026-05-05T08:36:21+00:00', 'title': 'RAG (Retrieval-Augmented Generation) Mastery R&D Guideline', 'moddate': '2026-05-05T08:36:21+00:00', 'source': '../data/raws/RAG_(Retrieval-Augmented_Generation)_Mastery_RD_Guideline.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'preprocessing': 'cleaned', 'cleaning_status': 'completed', 'original_char_count': 1377, 'cleaned_char_count': 1370, 'cleaned_word_count': 188, 'source_file': 'RAG_(Retrieval-Augmented_Generation)_Mastery_RD_Guideline.pdf', 'source_path': '../data/raws/RAG_(Retrieval-Augmented_Generation)_Mastery_RD_Guideline.pdf', 'page_index': 0, 'page_number': 1, 'document_type': 'pdf', 'pipeline_stage': 'step

In [31]:
import json

final_chunks_path = "../data/json/step2_final_processed_chunks.jsonl"

with open(final_chunks_path, "w", encoding="utf-8") as f:
    for chunk in final_processed_chunks:
        record = {
            "page_content": chunk.page_content,
            "metadata": chunk.metadata
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Final processed chunks saved to:")
print(final_chunks_path)

Final processed chunks saved to:
../data/json/step2_final_processed_chunks.jsonl


In [32]:
step2_report = f"""
# Step 2: Data Preprocessing Techniques for RAG

## Objective

The objective of Step 2 is to experiment with preprocessing techniques for RAG, including text cleaning, metadata extraction, chunking strategies, and overlap strategies.

## Cleaning

I cleaned the extracted PDF text by removing broken line-break hyphenation, unnecessary newlines, control characters, and repeated spaces.

## Metadata Extraction

I enriched each cleaned page with structured metadata, including source file, source path, page index, page number, document type, character count, word count, and preprocessing stage.

## Chunking Strategies Tested

I tested the following seven chunking strategies:

1. Page-level chunking
2. Fixed-size character chunking
3. Recursive character chunking
4. Sentence-based chunking
5. Semantic chunking
6. Hierarchical chunking
7. LLM-based chunking

## Chunking Summary

{chunking_summary.to_string(index=False)}

## Pros and Cons

{pros_cons_table.to_string(index=False)}

## Overlap Strategy Experiment

I tested multiple overlap values for recursive character chunking: 0, 80, 120, and 200.

{overlap_summary.to_string(index=False)}

## Selected Final Pipeline

- Cleaning function: clean_text()
- Metadata extraction: enrich_page_metadata()
- Chunking strategy: {selected_strategy_name}
- Chunk size: {selected_chunk_size}
- Chunk overlap: {selected_chunk_overlap}

## Reason for Selection

I selected recursive character chunking with chunk_size=800 and chunk_overlap=120 because it provides a good balance between chunk size control, context preservation, stability, and reproducibility.

Compared with fixed-size chunking, it preserves document structure better. Compared with page-level chunking, it avoids overly large chunks. Compared with LLM-based chunking, it is more stable, cost-effective, and easier to reproduce.

## Output

Final processed chunks were saved to:

{final_chunks_path}

## Next Step

Proceed to Step 3: Embedding Methods – Research & Selection.
"""

step2_report_path = "../data/artifacts/step2_preprocessing_report.md"

with open(step2_report_path, "w", encoding="utf-8") as f:
    f.write(step2_report)

print("Step 2 report saved to:")
print(step2_report_path)

Step 2 report saved to:
../data/artifacts/step2_preprocessing_report.md


# SAVE ARTIFACT

In [33]:
import json
import os

# Đường dẫn lưu trữ (Đảm bảo thư mục này đã tồn tại)
artifact_path = "../data/json/step2_final_processed_chunks.jsonl"
os.makedirs(os.path.dirname(artifact_path), exist_ok=True)

# Sử dụng đúng tên biến final_processed_chunks
with open(artifact_path, "w", encoding="utf-8") as f:
    for chunk in final_processed_chunks:
        record = {
            "page_content": chunk.page_content,
            "metadata": chunk.metadata
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Đã lưu thành công {len(final_processed_chunks)} chunks vào: {artifact_path}")

Đã lưu thành công 16 chunks vào: ../data/json/step2_final_processed_chunks.jsonl


In [34]:
# Lưu vào thư mục artifacts
vectorstore.save_local("../data/artifacts/faiss_index")
print("Đã lưu FAISS Index thành công!")

Đã lưu FAISS Index thành công!


# Nạp lại db

In [35]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.load_local(
    "../data/artifacts/faiss_index", 
    embeddings, 
    allow_dangerous_deserialization=True # Cần thiết khi load file local
)
print("Đã nạp lại Vector Database, sẵn sàng truy vấn!")

Đã nạp lại Vector Database, sẵn sàng truy vấn!


In [36]:
import json
import time
import gc
import pandas as pd
from langchain_core.documents import Document

# 1. Nạp lại mốc dữ liệu đã xử lý từ Bước 2
artifact_path = "../data/json/step2_final_processed_chunks.jsonl"

with open(artifact_path, "r", encoding="utf-8") as f:
    loaded_data = [json.loads(line) for line in f]

all_chunks = [Document(page_content=item["page_content"], metadata=item["metadata"]) for item in loaded_data]

# 2. Lấy 15 chunk ngẫu nhiên hoặc tiêu biểu làm tập mẫu benchmark
sample_chunks = all_chunks[:15]
sample_texts = [chunk.page_content for chunk in sample_chunks]

print(f"Đã nạp {len(all_chunks)} chunks tổng. Sử dụng {len(sample_texts)} chunks để Benchmark.")

Đã nạp 16 chunks tổng. Sử dụng 15 chunks để Benchmark.


In [37]:
def benchmark_embedding_model(model_name, embedding_instance, texts):
    print(f"\nĐang kiểm thử: {model_name}...")
    
    # Đo thời gian bắt đầu
    start_time = time.time()
    
    # Thực hiện nhúng hàng loạt (batch embedding)
    embeddings = embedding_instance.embed_documents(texts)
    
    # Đo thời gian kết thúc
    latency = time.time() - start_time
    time_per_chunk = (latency / len(texts)) * 1000 # tính bằng ms
    
    # Kích thước số chiều (Dimension)
    dimension = len(embeddings[0])
    
    print(f"Xong! Số chiều: {dimension} | Tổng thời gian: {latency:.2f}s | Độ trễ/chunk: {time_per_chunk:.2f}ms")
    return {
        "Model": model_name,
        "Dimension": dimension,
        "Total Latency (s)": round(latency, 2),
        "Latency per Chunk (ms)": round(time_per_chunk, 2),
        "Hosting Type": "Cloud (API)" if "Gemini" in model_name else "Local (Self-hosted)"
    }

# Danh sách lưu kết quả
results = []

In [38]:
import os
import gc
import vertexai
from dotenv import load_dotenv
from langchain_google_vertexai import VertexAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

# 0. Nạp Token bảo mật từ file .env
load_dotenv()
hf_token = os.getenv("HF_API_KEY")

# Khởi tạo kết nối GCP cho Gemini
vertexai.init(project="rag-project-495601", location="us-central1")

# 1. GEMINI EMBEDDING 2 (Cloud API)
gemini_model = VertexAIEmbeddings(model_name="text-embedding-004")
res = benchmark_embedding_model("Gemini Embedding 2", gemini_model, sample_texts)
results.append(res)
del gemini_model
gc.collect()

# 2. MULTILINGUAL-E5-LARGE
e5_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    model_kwargs={
        'device': 'cpu',
        'token': hf_token
    }, 
    encode_kwargs={'normalize_embeddings': True}
)
res = benchmark_embedding_model("multilingual-e5-large", e5_model, sample_texts)
results.append(res)

del e5_model
gc.collect()

# 3. BGE-M3 (Local - Chuyên gia đa ngôn ngữ)
bgem3_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={
        'device': 'cpu',
        'token': hf_token
    },
    encode_kwargs={'normalize_embeddings': True}
)
res = benchmark_embedding_model("BGE-M3", bgem3_model, sample_texts)
results.append(res)

del bgem3_model
gc.collect()

C:\Users\XuanDongDev\AppData\Local\Temp\ipykernel_33624\116716660.py:16: DeprecationWarning: Use [`GoogleGenerativeAIEmbeddings`][langchain_google_genai.GoogleGenerativeAIEmbeddings] instead.
  gemini_model = VertexAIEmbeddings(model_name="text-embedding-004")
C:\Users\XuanDongDev\AppData\Local\Temp\ipykernel_33624\116716660.py:16: LangChainDeprecationWarning: The class `VertexAIEmbeddings` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import GoogleGenerativeAIEmbeddings``.
  gemini_model = VertexAIEmbeddings(model_name="text-embedding-004")



Đang kiểm thử: Gemini Embedding 2...
Xong! Số chiều: 768 | Tổng thời gian: 2.53s | Độ trễ/chunk: 168.58ms


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6647.94it/s]



Đang kiểm thử: multilingual-e5-large...
Xong! Số chiều: 1024 | Tổng thời gian: 9.39s | Độ trễ/chunk: 626.09ms


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 29093.01it/s]



Đang kiểm thử: BGE-M3...
Xong! Số chiều: 1024 | Tổng thời gian: 10.66s | Độ trễ/chunk: 710.97ms


7857

In [39]:
df_benchmark = pd.DataFrame(results)

print("\nMA TRẬN SO SÁNH HIỆU NĂNG EMBEDDING (BƯỚC 3):")
display(df_benchmark)

df_benchmark.to_csv("../data/artifacts/step3_embedding_benchmark.csv", index=False)
print("Đã lưu ma trận benchmark vào './data/artifacts/step3_embedding_benchmark.csv'")


MA TRẬN SO SÁNH HIỆU NĂNG EMBEDDING (BƯỚC 3):


,Model,Dimension,Total Latency (s),Latency per Chunk (ms),Hosting Type
0,Gemini Embedding 2,768,2.53,168.58,Cloud (API)
1,multilingual-e5-large,1024,9.39,626.09,Local (Self-hosted)
2,BGE-M3,1024,10.66,710.97,Local (Self-hosted)


Đã lưu ma trận benchmark vào './data/artifacts/step3_embedding_benchmark.csv'
